In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
PROJECT_ID = "project-7f16ff1d-9026-4799-bfa"
BUCKET = "amazon-reviews-lakehouse-data"
DATASET_ID = "gold_zone"
SOURCE_TABLE = "product_user_mapping"
TARGET_TABLE = "user_product_training_data"
TEMP_GCS_BUCKET = BUCKET

spark = DataprocSparkSession.builder.getOrCreate()

████████████████████████▌                                                       

In [ ]:
def build_gold_training_data_ultimate():
    print(f" Bắt đầu quy trình...")
    df_pos = spark.read.format("bigquery").load(f"{PROJECT_ID}.{DATASET_ID}.{SOURCE_TABLE}")
    df_pos_labeled = df_pos.withColumn("label", F.lit(1))

    # ---------------------------------------------------------
    # BƯỚC 1: LẤY 1000 SẢN PHẨM NGẪU NHIÊN LÀM "KHO TẠM"
    # ---------------------------------------------------------
    print(" Đang tạo kho 1000 sản phẩm ngẫu nhiên...")
    # Lấy 1000 mã sản phẩm duy nhất đưa về máy khách (rất nhẹ)
    sample_rows = df_pos.select("parent_asin").distinct().limit(1000).collect()
    sample_products = [row[0] for row in sample_rows]

    # Biến danh sách đó thành một Cột dạng Array của Spark
    array_col = F.array([F.lit(x) for x in sample_products])

    # ---------------------------------------------------------
    # BƯỚC 2: TẠO NHÃN 0 BẰNG HÀM MẢNG (SIÊU NHANH)
    # ---------------------------------------------------------
    print(" Đang sinh dữ liệu Negative...")
    # Gom nhóm số lượng đã mua
    df_user_history = df_pos.groupBy("user_id").agg(
        F.collect_set("parent_asin").alias("bought_products"),
        F.count("parent_asin").alias("num_bought")
    )

    # Lấy "Kho tạm 1000" TRỪ ĐI "Đã mua", sau đó Xáo trộn và Cắt đúng số lượng
    df_neg_arrays = df_user_history.withColumn(
        "unbought_products", F.array_except(array_col, F.col("bought_products"))
    ).withColumn(
        "neg_products", F.slice(F.shuffle("unbought_products"), 1, F.col("num_bought"))
    )

    # Trải phẳng mảng ra thành các hàng
    df_neg_final = df_neg_arrays.select(
        "user_id",
        F.explode("neg_products").alias("parent_asin")
    ).withColumn("label", F.lit(0))

    # ---------------------------------------------------------
    # BƯỚC 3: GỘP VÀ LƯU VÀO BIGQUERY
    # ---------------------------------------------------------
    print(" Đang gộp dữ liệu và lưu vào BigQuery...")
    final_df = df_pos_labeled.select("user_id", "parent_asin", "label") \
        .unionByName(df_neg_final)

    final_df.write.format("bigquery") \
        .option("table", f"{PROJECT_ID}.{DATASET_ID}.{TARGET_TABLE}") \
        .option("temporaryGcsBucket", TEMP_GCS_BUCKET) \
        .mode("overwrite") \
        .save()

    print(" HOÀN TẤT! Đã vượt qua ải dữ liệu lớn thành công.")
    final_df.show(5)

In [ ]:
if __name__ == "__main__":
    build_gold_training_data_ultimate()

 Bắt đầu quy trình...
 Đang tạo kho 1000 sản phẩm ngẫu nhiên...


  0%|           0/169 Tasks

 Đang sinh dữ liệu Negative...
 Đang gộp dữ liệu và lưu vào BigQuery...


  0%|           0/592 Tasks

 HOÀN TẤT! Đã vượt qua ải dữ liệu lớn thành công.


  0%|           0/592 Tasks

+--------------------+-----------+-----+
|             user_id|parent_asin|label|
+--------------------+-----------+-----+
|AFHMASYWKFSJQ73CY...| 0001473727|    1|
|AFDP3ZNMHEWLPPO2D...| 0002154463|    1|
|AGKMRGMZ7Y2O2MU24...| 0002214482|    1|
|AGB4HENMT26ZBRLAO...| 0003303209|    1|
|AGP7TWGCQMXCL3IIF...| 0005164885|    1|
+--------------------+-----------+-----+
only showing top 5 rows
